# Data Curation with RDKit

Pipeline:
1. Load the KNIME CSV
2. Parse SMILES with RDKit
3. Remove salts (keep largest organic fragment)
4. Add hydrogens
5. Neutralise formal charges
6. Standardise molecules (normalise functional groups, tautomer consistency)
7. Filter out inorganic compounds
8. Save curated data

In [2]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import MolStandardize
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.rdMolDescriptors import CalcMolFormula

---
## 1. Load data

In [3]:
df = pd.read_csv('the_final_data_base_in_KNIME.csv')
print(f'Rows loaded: {len(df)}')
df[['CAS_str', 'Chemical_name', 'SMILES']].head()

Rows loaded: 2375


,CAS_str,Chemical_name,SMILES
0,50-06-6,Phenobarbital,CCC1(C(=O)NC(=O)NC1=O)C2=CC=CC=C2
1,50-18-0,Cyclophosphamide,C1CNP(=O)(OC1)N(CCCl)CCCl
2,50-28-2,17beta-Estradiol,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)CCC4=C3...
3,50-29-3,"p,p'-DDT",C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)(Cl)Cl)Cl
4,50-30-6,"2,6-Dichlorobenzoic acid",C1=CC(=C(C(=C1)Cl)C(=O)O)Cl


In [4]:
# Check for missing SMILES
print(f'Missing SMILES: {df["SMILES"].isna().sum()}')
df = df.dropna(subset=['SMILES']).reset_index(drop=True)
print(f'Rows after dropping missing SMILES: {len(df)}')

Missing SMILES: 0
Rows after dropping missing SMILES: 2375


---
## 2. Parse SMILES with RDKit

In [5]:
def parse_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol
    except:
        return None

df['mol'] = df['SMILES'].apply(parse_smiles)

n_valid = df['mol'].notna().sum()
n_fail = df['mol'].isna().sum()
print(f'Valid molecules: {n_valid}')
print(f'Unparseable SMILES: {n_fail}')

df = df.dropna(subset=['mol']).reset_index(drop=True)

Valid molecules: 2375
Unparseable SMILES: 0


[18:28:28] WARNING: not removing hydrogen atom without neighbors
/var/folders/wc/y79sy_bd62b05zwn7tgfb2s40000gq/T/ipykernel_48372/3540612574.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['mol'] = df['SMILES'].apply(parse_smiles)


---
## 3. Remove salts (keep largest fragment)

In [6]:
def remove_salt(mol):
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    if len(frags) == 1:
        return mol
    largest = max(frags, key=lambda m: m.GetNumAtoms())
    try:
        Chem.SanitizeMol(largest)
        return largest
    except:
        return mol

n_before = df['mol'].apply(lambda m: len(Chem.GetMolFrags(m)) > 1).sum()
df['mol'] = df['mol'].apply(remove_salt)
print(f'Molecules with multiple fragments (salts) removed: {n_before}')
still_multi = df['mol'].apply(lambda m: len(Chem.GetMolFrags(m)) > 1).sum()
print(f'Still multi-fragment after clean: {still_multi}')

Molecules with multiple fragments (salts) removed: 330
Still multi-fragment after clean: 0


---
## 4. Add hydrogens

In [7]:
df['mol'] = df['mol'].apply(Chem.AddHs)
print('Added explicit hydrogens to all molecules')

Added explicit hydrogens to all molecules


---
## 5. Neutralise formal charges

In [8]:
def uncharger(mol):
    try:
        return rdMolStandardize.Uncharger().uncharge(mol)
    except:
        return mol

n_charged_before = sum(1 for m in df['mol'] if any(at.GetFormalCharge() != 0 for at in m.GetAtoms()))
df['mol'] = df['mol'].apply(uncharger)
n_charged_after = sum(1 for m in df['mol'] if any(at.GetFormalCharge() != 0 for at in m.GetAtoms()))
print(f'Molecules with formal charges before: {n_charged_before}')
print(f'Molecules with formal charges after:  {n_charged_after}')

[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Removed negative charge.
[18:28:42] Running Uncharger
[18:28:42] Removed negative charge.
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger


Molecules with formal charges before: 422
Molecules with formal charges after:  245


[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Removed negative charge.
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Removed negative charge.
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger
[18:28:42] Running Uncharger


---
## 6. Standardise molecules

Normalise functional groups and canonicalise tautomers using RDKit's `rdMolStandardize`.

In [9]:
normalizer = rdMolStandardize.Normalizer()
tautomerizer = rdMolStandardize.TautomerEnumerator()

def standardize(mol):
    try:
        mol = normalizer.normalize(mol)
        mol = tautomerizer.Canonicalize(mol)
        return mol
    except:
        return mol

df['mol'] = df['mol'].apply(standardize)
print('Standardisation complete (normalisation + tautomer canonicalisation).')

[18:28:51] Initializing Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[18:28:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2
[18:28:51] Can't kekulize mol.  Unkekulized atoms: 0 1 5
[18:28:51] Can't kekulize mol.  Unkekulized atoms: 0 1 5
[18:28:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2
[18:28:51] Can't kekulize mol.  Unkekulized atoms: 0 1 5
[18:28:51] Can't kekulize mol.  Unkekulized atoms: 0 1 5
[18:28:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2
[18:28:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2
[18:28:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2
[18:28:51] Running Normalizer
[18:28:51] Running Normalizer
[1

Standardisation complete (normalisation + tautomer canonicalisation).


[18:29:10] Running Normalizer
[18:29:10] Running Normalizer
[18:29:10] Running Normalizer
[18:29:10] Running Normalizer
[18:29:10] Running Normalizer


---
## 7. Filter out inorganic compounds

Definition: a molecule is **inorganic** if it:
- Contains **no carbon atoms**, OR
- Contains **heavy atoms outside the organic set** (e.g. metals)

Both filters are shown below.

In [10]:
# ---- Filter definitions ----

organic_set = {'H', 'B', 'C', 'N', 'O', 'F', 'Si', 'P', 'S', 'Cl', 'Se', 'Br', 'I'}

def has_carbon(mol):
    return any(at.GetAtomicNum() == 6 for at in mol.GetAtoms())

def is_inorganic_no_carbon(mol):
    """Flag molecules with zero carbon atoms."""
    return not has_carbon(mol)

def is_inorganic_heavy(mol):
    """Flag molecules with atoms outside the organic set (e.g. metals)."""
    for at in mol.GetAtoms():
        if at.GetSymbol() not in organic_set:
            return True
    return False

# ---- Apply filters ----

df['inorganic_no_carbon'] = df['mol'].apply(is_inorganic_no_carbon)
df['inorganic_heavy_atom'] = df['mol'].apply(is_inorganic_heavy)
df['inorganic'] = df['inorganic_no_carbon'] | df['inorganic_heavy_atom']

n_inorganic = df['inorganic'].sum()
print('=== INORGANIC FILTER RESULTS ===')
print(f'Filter 1 - No carbon:               {df["inorganic_no_carbon"].sum()}')
print(f'Filter 2 - Non-organic heavy atom:   {df["inorganic_heavy_atom"].sum()}')
print(f'Total flagged as inorganic:           {n_inorganic}')
print(f'Organic molecules retained:            {len(df) - n_inorganic}')

=== INORGANIC FILTER RESULTS ===
Filter 1 - No carbon:               25
Filter 2 - Non-organic heavy atom:   58
Total flagged as inorganic:           72
Organic molecules retained:            2303


/var/folders/wc/y79sy_bd62b05zwn7tgfb2s40000gq/T/ipykernel_48372/3549395085.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['inorganic_no_carbon'] = df['mol'].apply(is_inorganic_no_carbon)
/var/folders/wc/y79sy_bd62b05zwn7tgfb2s40000gq/T/ipykernel_48372/3549395085.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['inorganic_heavy_atom'] = df['mol'].apply(is_inorganic_heavy)
/var/folders/wc/y79sy_bd62b05zwn7tgfb2s40000gq/T/ipykernel_48372/3549395085.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is

In [11]:
# ---- Display flagged molecules ----

inorganic_df = df[df['inorganic']][['CAS_str', 'Chemical_name', 'SMILES']].copy()

print(f'--- {len(inorganic_df)} molecules flagged as inorganic ---')
if len(inorganic_df) > 0:
    formula_map = dict(zip(range(len(df)), (CalcMolFormula(m) for m in df['mol'])))
    inorganic_df = inorganic_df.copy()
    inorganic_df['Formula'] = [formula_map[i] for i in df[df['inorganic']].index]
    display(inorganic_df)
else:
    print('No inorganic molecules found.')

--- 72 molecules flagged as inorganic ---


,CAS_str,Chemical_name,SMILES,Formula
24,54-64-8,Thimerosal,CC[Hg]SC1=CC=CC=C1C(=O)[O-].[Na+],C9H10HgO2S
33,56-35-9,Bis(tributyltin)oxide,CCCC[Sn](CCCC)(CCCC)O[Sn](CCCC)(CCCC)CCCC,C24H54OSn2
34,56-36-0,Tributyltin acetate,CCCC[Sn](CCCC)(CCCC)OC(=O)C,C14H30O2Sn
68,62-38-4,Phenylmercuric acetate,CC(=O)O[Hg]C1=CC=CC=C1,C8H8HgO2
104,75-60-5,Dimethylarsinic acid,C[As](=O)(C)O,C2H7AsO2
...,...,...,...,...
1797,16903-35-8,Tetrachloroauric acid,[H+].[Cl-].[Cl-].[Cl-].[Cl-].[Au+3],H+
2002,37222-66-5,Potassium peroxymonosulfate sulfate,OS(=O)(=O)[O-].OS(=O)(=O)O[O-].OS(=O)(=O)O[O-]...,H2O5S
2372,Metalgrp.CrIII,Chromium chloride (CrCl3),Cl[Cr](Cl)Cl,Cl3Cr
2373,Metalgrp.Hg,Mercuric chloride,Cl[Hg]Cl,Cl2Hg


In [12]:
# ---- Remove the inorganic molecules ----

df_organic = df[~df['inorganic']].copy().reset_index(drop=True)
print(f'After removing inorganic: {len(df_organic)} molecules remaining')

After removing inorganic: 2303 molecules remaining


---
## 8. Generate final SMILES and save

In [13]:
# Write back the curated SMILES
df_organic['SMILES_curated'] = df_organic['mol'].apply(
    lambda m: Chem.MolToSmiles(m, canonical=True) if m is not None else None
)

# Drop the helper columns and the mol object
cols_to_drop = ['mol', 'inorganic_no_carbon', 'inorganic_heavy_atom', 'inorganic']
cols_present = [c for c in cols_to_drop if c in df_organic.columns]
df_out = df_organic.drop(columns=cols_present)

print(f'Final dataset shape: {df_out.shape}')
df_out.head()

Final dataset shape: (2303, 126)


,CAS_str,Chemical_name,log_LC50,n_measurements,SMILES,mlecule_H_final,SlogP,SMR,LabuteASA,TPSA,...,MQN34,MQN35,MQN36,MQN37,MQN38,MQN39,MQN40,MQN41,MQN42,SMILES_curated
0,50-06-6,Phenobarbital,2.685294,2,CCC1(C(=O)NC(=O)NC1=O)C2=CC=CC=C2,[H]c1c([H])c([H])c(C2(C([H])([H])C([H])([H])[H...,0.7004,60.0924,115.175275,75.27,...,0,0,2,0,0,0,0,0,0,[H]c1c([H])c([H])c(C2(C([H])([H])C([H])([H])[H...
1,50-18-0,Cyclophosphamide,3.340444,1,C1CNP(=O)(OC1)N(CCCl)CCCl,[H]N1C([H])([H])C([H])([H])C([H])([H])OP1(=O)N...,1.8840,59.1912,115.574149,41.57,...,0,0,1,0,0,0,0,0,0,[H]N1C([H])([H])C([H])([H])C([H])([H])OP1(=O)N...
2,50-28-2,17beta-Estradiol,0.330673,5,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)CCC4=C3...,[H]Oc1c([H])c([H])c2c(c1[H])C([H])([H])C([H])(...,3.6092,78.7306,154.274315,40.46,...,0,1,3,0,0,0,0,6,3,[H]O[C@@]1([H])C([H])([H])C([H])([H])[C@@]2([H...
3,50-29-3,"p,p'-DDT",-1.737289,448,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)(Cl)Cl)Cl,[H]c1c([H])c(C([H])(c2c([H])c([H])c(Cl)c([H])c...,6.4955,85.0370,149.379447,0.00,...,0,0,2,0,0,0,0,0,0,[H]c1c([H])c(C([H])(c2c([H])c([H])c(Cl)c([H])c...
4,50-30-6,"2,6-Dichlorobenzoic acid",2.112655,6,C1=CC(=C(C(=C1)Cl)C(=O)O)Cl,[H]OC(=O)c1c(Cl)c([H])c([H])c([H])c1Cl,2.6916,43.4213,79.065186,37.30,...,0,0,1,0,0,0,0,0,0,[H]O=C(O)c1c(Cl)c([H])c([H])c([H])c1Cl


In [14]:
output_path = 'data/curated_data_rdkit.csv'
df_out.to_csv(output_path, index=False)
print(f'Saved to {output_path}')

Saved to data/curated_data_rdkit.csv


In [15]:
# Quick summary
print('--- Summary ---')
print(f'Original rows:                     2375')
print(f'After dropping missing SMILES:     {len(df)}')
print(f'After removing inorganic:          {len(df_organic)}')
print(f'Dropped (inorganic):               {n_inorganic}')
print(f'Final columns:                     {list(df_out.columns)}')

--- Summary ---
Original rows:                     2375
After dropping missing SMILES:     2375
After removing inorganic:          2303
Dropped (inorganic):               72
Final columns:                     ['CAS_str', 'Chemical_name', 'log_LC50', 'n_measurements', 'SMILES', 'mlecule_H_final', 'SlogP', 'SMR', 'LabuteASA', 'TPSA', 'AMW', 'ExactMW', 'NumLipinskiHBA', 'NumLipinskiHBD', 'NumRotatableBonds', 'NumHBD', 'NumHBA', 'NumAmideBonds', 'NumHeteroAtoms', 'NumHeavyAtoms', 'NumAtoms', 'NumStereocenters', 'NumUnspecifiedStereocenters', 'NumRings', 'NumAromaticRings', 'NumSaturatedRings', 'NumAliphaticRings', 'NumAromaticHeterocycles', 'NumSaturatedHeterocycles', 'NumAliphaticHeterocycles', 'NumAromaticCarbocycles', 'NumSaturatedCarbocycles', 'NumAliphaticCarbocycles', 'FractionCSP3', 'Chi0v', 'Chi1v', 'Chi2v', 'Chi3v', 'Chi4v', 'Chi1n', 'Chi2n', 'Chi3n', 'Chi4n', 'HallKierAlpha', 'kappa1', 'kappa2', 'kappa3', 'slogp_VSA1', 'slogp_VSA2', 'slogp_VSA3', 'slogp_VSA4', 'slogp_VSA5', 'slog